In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Users\danie\supportmind-ai\rawsupportminddata\Help Desk Tickets\Help Desk Tickets"
)

issues = pd.read_csv(DATA_DIR / "issues.csv")



print("issues:", issues.shape)



issues: (66691, 58)


In [3]:
print("Priority distribution")
print("=" * 50)

priority_counts = issues["issue_priority"].value_counts(dropna=False)

print(priority_counts)

print("\nPriority percentages")
print("=" * 50)

priority_pct = (
    issues["issue_priority"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

print(priority_pct)

print("\nTarget diagnostics")
print("=" * 50)

print("Missing priority:", issues["issue_priority"].isna().sum())
print("Unique priority values:", issues["issue_priority"].nunique(dropna=False))
print("\nValues:")
print(issues["issue_priority"].unique())

Priority distribution
issue_priority
unknown    33965
Medium     24788
High        4554
Highest     2084
Blocker      656
Low          560
Lowest        84
Name: count, dtype: int64

Priority percentages
issue_priority
unknown    50.93
Medium     37.17
High        6.83
Highest     3.12
Blocker     0.98
Low         0.84
Lowest      0.13
Name: proportion, dtype: float64

Target diagnostics
Missing priority: 0
Unique priority values: 7

Values:
<ArrowStringArray>
['Medium', 'High', 'unknown', 'Low', 'Blocker', 'Highest', 'Lowest']
Length: 7, dtype: str


In [4]:
# Priority target and issue-level structure

print("Priority distribution:")
print(issues["issue_priority"].value_counts(dropna=False))

print("\nUnique issue_num:", issues["issue_num"].nunique())
print("Total rows:", len(issues))

rows_per_issue = issues.groupby("issue_num").size()

print("\nRows per issue:")
print(rows_per_issue.describe())

print("\nIssues with >1 row:", (rows_per_issue > 1).sum())

Priority distribution:
issue_priority
unknown    33965
Medium     24788
High        4554
Highest     2084
Blocker      656
Low          560
Lowest        84
Name: count, dtype: int64

Unique issue_num: 3542
Total rows: 66691

Rows per issue:
count    3542.000000
mean       18.828628
std        32.729260
min         1.000000
25%         3.000000
50%         5.000000
75%        20.000000
max       269.000000
dtype: float64

Issues with >1 row: 3328


In [5]:
# Keep only rows with a real priority label
priority_df = issues[
    issues["issue_priority"].ne("unknown")
].copy()

print("Priority modelling dataset:", priority_df.shape)

print("\nTarget distribution")
print("=" * 50)
print(priority_df["issue_priority"].value_counts())

print("\nFeature columns")
print("=" * 50)

for i, col in enumerate(priority_df.columns, start=1):
    print(f"{i:02d}. {col}")

Priority modelling dataset: (32726, 58)

Target distribution
issue_priority
Medium     24788
High        4554
Highest     2084
Blocker      656
Low          560
Lowest        84
Name: count, dtype: int64

Feature columns
01. id
02. started
03. ended
04. issue_num
05. issue_proj
06. issue_reporter
07. issue_assignee
08. issue_contr_count
09. issue_type
10. issue_priority
11. issue_created
12. issue_resolution_date
13. issue_resolution
14. issue_status
15. issue_comments_count
16. last_change_date
17. wf_in_review
18. wfe_in_review
19. wf_deployment
20. wfe_deployment
21. wf_resolved
22. wfe_resolved
23. wf_open
24. wfe_open
25. wf_monitoring
26. wfe_monitoring
27. wf_done
28. wfe_done
29. wf_pending_customer_approval
30. wfe_pending_customer_approval
31. wf_rejected
32. wfe_rejected
33. wf_testing_monitoring
34. wfe_testing_monitoring
35. wf_in_progress
36. wfe_in_progress
37. wf_reopened
38. wfe_reopened
39. wf_to_do
40. wfe_to_do
41. wf_validation
42. wfe_validation
43. wf_resolved_un

In [6]:
print("\nColumn diagnostics")
print("=" * 50)

diagnostics = pd.DataFrame({
    "dtype": priority_df.dtypes.astype(str),
    "unique": priority_df.nunique(dropna=False),
    "missing": priority_df.isna().sum(),
    "missing_pct": (
        priority_df.isna().mean() * 100
    ).round(2),
})

print(diagnostics.to_string())


Column diagnostics
                                 dtype  unique  missing  missing_pct
id                             float64   32726        0         0.00
started                            str   32344        0         0.00
ended                              str   32533        0         0.00
issue_num                      float64    3466        0         0.00
issue_proj                         str     331        0         0.00
issue_reporter                     str     788        0         0.00
issue_assignee                     str     300     7775        23.76
issue_contr_count              float64       9        0         0.00
issue_type                         str      15        0         0.00
issue_priority                     str       6        0         0.00
issue_created                      str   32344        0         0.00
issue_resolution_date              str   31678      853         2.61
issue_resolution                   str       5      853         2.61
issue_status  

In [7]:
# Check whether the same issue_num has multiple priority labels
priority_consistency = (
    priority_df.groupby("issue_num")["issue_priority"]
    .nunique()
    .sort_values(ascending=False)
)

print("Unique issues:", priority_df["issue_num"].nunique())
print("Issues with >1 priority:", (priority_consistency > 1).sum())
print("Maximum priorities for one issue:", priority_consistency.max())

print("\nIssues with multiple priorities:")
print(priority_consistency[priority_consistency > 1].head(20))

Unique issues: 3466
Issues with >1 priority: 2348
Maximum priorities for one issue: 6

Issues with multiple priorities:
issue_num
30.0     6
20.0     6
11.0     6
15.0     6
64.0     6
2.0      6
114.0    6
71.0     6
62.0     6
201.0    6
210.0    6
48.0     6
28.0     5
43.0     5
42.0     5
27.0     5
44.0     5
31.0     5
32.0     5
8.0      5
Name: issue_priority, dtype: int64


In [8]:
# Number of rows per issue
issue_row_counts = priority_df["issue_num"].value_counts()

print("Rows per unique issue")
print(issue_row_counts.describe())

print("\nTop issues by row count")
print(issue_row_counts.head(20))

Rows per unique issue
count    3466.000000
mean        9.442008
std        16.486672
min         1.000000
25%         2.000000
50%         3.000000
75%         6.000000
max       105.000000
Name: count, dtype: float64

Top issues by row count
issue_num
2.0     105
6.0     103
4.0     101
3.0     101
8.0      99
15.0     99
5.0      99
11.0     99
10.0     98
19.0     97
17.0     97
14.0     97
18.0     96
7.0      96
13.0     96
9.0      96
12.0     94
16.0     94
20.0     93
24.0     92
Name: count, dtype: int64


In [9]:
# Priority distribution by issue type
priority_by_type = pd.crosstab(
    priority_df["issue_type"],
    priority_df["issue_priority"],
    normalize="index"
).round(3)

print(priority_by_type)

issue_priority  Blocker   High  Highest    Low  Lowest  Medium
issue_type                                                    
Assistance        0.000  0.111    0.074  0.019   0.000   0.796
Bug               0.000  0.019    0.000  0.000   0.000   0.981
Deployment        0.067  0.225    0.129  0.020   0.000   0.558
Epic              0.000  0.005    0.000  0.002   0.000   0.993
HD Service        0.002  0.109    0.042  0.010   0.006   0.832
Project           0.004  0.093    0.036  0.012   0.002   0.853
Retrospective     0.008  0.008    0.000  0.000   0.000   0.983
Service           0.006  0.098    0.040  0.022   0.004   0.829
Sprint Summary    0.000  0.029    0.019  0.005   0.000   0.947
Story             0.006  0.067    0.017  0.011   0.003   0.895
Sub-task          0.002  0.061    0.013  0.006   0.009   0.910
Subtask           0.000  0.000    0.000  0.000   0.000   1.000
Task              0.002  0.081    0.018  0.003   0.002   0.894
Ticket            0.031  0.185    0.091  0.020   0.001 

In [10]:
# Inspect how priority changes over the lifecycle of a few issues

multi_priority_issues = (
    priority_df.groupby("issue_num")["issue_priority"]
    .nunique()
)

sample_issue_ids = (
    multi_priority_issues[multi_priority_issues > 1]
    .head(5)
    .index
)

cols = [
    "issue_num",
    "started",
    "ended",
    "issue_created",
    "last_change_date",
    "issue_priority",
    "issue_status",
    "issue_type",
    "processing_steps",
]

sample_history = (
    priority_df[
        priority_df["issue_num"].isin(sample_issue_ids)
    ][cols]
    .sort_values(["issue_num", "started"])
)

print(sample_history.to_string(index=False))

 issue_num                          started                            ended                    issue_created                 last_change_date issue_priority issue_status    issue_type  processing_steps
       1.0        2015-08-12 11:03:33+00:00        2018-01-23 14:50:06+00:00        2015-08-12 11:03:33+00:00        2018-01-23 14:50:06+00:00         Medium         done          Epic                 1
       1.0        2015-08-13 06:04:08+00:00 2018-03-25 07:37:18.945000+00:00        2015-08-13 06:04:08+00:00 2018-03-25 07:37:18.945000+00:00         Medium         done          Epic                 1
       1.0        2015-08-13 06:39:24+00:00        2017-09-26 05:51:32+00:00        2015-08-13 06:39:24+00:00        2017-09-26 05:51:32+00:00         Medium         done          Epic                 1
       1.0        2015-08-13 14:19:00+00:00        2018-01-23 14:48:44+00:00        2015-08-13 14:19:00+00:00 2019-02-17 14:01:57.524000+00:00         Medium         done          Epic    

In [11]:
# Check whether issue_proj + issue_num uniquely identifies an issue

key_counts = (
    priority_df
    .groupby(["issue_proj", "issue_num"])
    .size()
    .sort_values(ascending=False)
)

duplicate_keys = key_counts[key_counts > 1]

print("Total project + issue_num keys:", len(key_counts))
print("Keys appearing more than once:", len(duplicate_keys))
print("Maximum rows for one project + issue_num:", key_counts.max())

print("\nTop repeated keys:")
print(duplicate_keys.head(20))

Total project + issue_num keys: 32665
Keys appearing more than once: 4
Maximum rows for one project + issue_num: 43

Top repeated keys:
issue_proj             issue_num
C06hg2xcu              1923.0       43
                       1922.0       18
C07ggu_0               143.0         2
C12hhx18aj28av.ew8x3d  34.0          2
dtype: int64


In [12]:
# Check whether the same project + issue_num has multiple priorities

priority_consistency = (
    priority_df
    .groupby(["issue_proj", "issue_num"])["issue_priority"]
    .nunique()
)

print(
    "Project + issue_num keys with multiple priorities:",
    (priority_consistency > 1).sum()
)

print(
    "Maximum number of priorities for one project + issue_num:",
    priority_consistency.max()
)

Project + issue_num keys with multiple priorities: 0
Maximum number of priorities for one project + issue_num: 1


In [13]:
duplicate_keys = (
    priority_df.groupby(["issue_proj", "issue_num"])
    .size()
)

duplicate_keys = duplicate_keys[duplicate_keys > 1]

duplicate_rows = (
    priority_df[
        priority_df.set_index(["issue_proj", "issue_num"])
        .index.isin(duplicate_keys.index)
    ]
    .sort_values(["issue_proj", "issue_num", "issue_created"])
)

print(duplicate_rows[
    [
        "issue_proj",
        "issue_num",
        "issue_created",
        "started",
        "issue_priority",
        "issue_type",
        "issue_comments_count",
        "processing_steps",
        "wf_total_time",
    ]
].to_string(index=False))

           issue_proj  issue_num             issue_created                   started issue_priority issue_type  issue_comments_count  processing_steps  wf_total_time
            C06hg2xcu     1922.0 2023-03-05 19:34:11+00:00 2023-03-05 19:34:11+00:00           High     Ticket                     2                 2        41700.0
            C06hg2xcu     1922.0 2023-03-05 19:34:11+00:00 2023-03-05 19:34:11+00:00           High     Ticket                     2                 1        41822.0
            C06hg2xcu     1922.0 2023-03-05 19:34:11+00:00 2023-03-05 19:34:11+00:00           High     Ticket                     4                 4        41925.0
            C06hg2xcu     1922.0 2023-03-05 19:34:12+00:00 2023-03-05 19:34:12+00:00           High     Ticket                     2                 1        42627.0
            C06hg2xcu     1922.0 2023-03-05 19:34:12+00:00 2023-03-05 19:34:12+00:00           High     Ticket                     2                 1        41612.0
    

In [14]:
# Remove the 4 issue keys that have multiple snapshot rows
key_cols = ["issue_proj", "issue_num"]

key_counts = (
    priority_df.groupby(key_cols)
    .size()
)

duplicate_keys = key_counts[key_counts > 1].index

duplicate_mask = (
    priority_df.set_index(key_cols)
    .index
    .isin(duplicate_keys)
)

priority_issue_df = priority_df.loc[~duplicate_mask].copy()

print("Original labelled rows :", len(priority_df))
print("Duplicate issue keys  :", len(duplicate_keys))
print("Rows removed          :", duplicate_mask.sum())
print("Clean issue rows      :", len(priority_issue_df))

print("\nPriority distribution")
print(priority_issue_df["issue_priority"].value_counts())

Original labelled rows : 32726
Duplicate issue keys  : 4
Rows removed          : 65
Clean issue rows      : 32661

Priority distribution
issue_priority
Medium     24786
High        4493
Highest     2084
Blocker      654
Low          560
Lowest        84
Name: count, dtype: int64


In [15]:
# Build the leakage-safe feature table for Priority Detection

model_df = priority_issue_df.copy()

# Parse issue creation timestamp
model_df["issue_created"] = pd.to_datetime(
    model_df["issue_created"],
    utc=True,
    errors="coerce",
)

# Derive creation-time features
model_df["created_year"] = model_df["issue_created"].dt.year
model_df["created_month"] = model_df["issue_created"].dt.month
model_df["created_dayofweek"] = model_df["issue_created"].dt.dayofweek
model_df["created_hour"] = model_df["issue_created"].dt.hour

# Keep only leakage-safe predictors
feature_cols = [
    "issue_proj",
    "issue_type",
    "created_year",
    "created_month",
    "created_dayofweek",
    "created_hour",
]

X = model_df[feature_cols].copy()
y = model_df["issue_priority"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nMissing values:")
print(X.isna().sum())

print("\nTarget:")
print(y.value_counts())

X shape: (32661, 6)
y shape: (32661,)

Features:
['issue_proj', 'issue_type', 'created_year', 'created_month', 'created_dayofweek', 'created_hour']

Missing values:
issue_proj               0
issue_type               0
created_year         12815
created_month        12815
created_dayofweek    12815
created_hour         12815
dtype: int64

Target:
issue_priority
Medium     24786
High        4493
Highest     2084
Blocker      654
Low          560
Lowest        84
Name: count, dtype: int64


In [16]:
# Inspect issue_created values that failed timestamp parsing

raw_created = priority_issue_df["issue_created"]

parsed_created = pd.to_datetime(
    raw_created,
    utc=True,
    errors="coerce",
    format="mixed",
)

invalid_mask = raw_created.notna() & parsed_created.isna()

print("Total rows:", len(raw_created))
print("Parsing failures:", invalid_mask.sum())

print("\nExamples of unparsed issue_created values:")
print(
    raw_created[invalid_mask]
    .drop_duplicates()
    .head(20)
    .to_string(index=False)
)

Total rows: 32661
Parsing failures: 0

Examples of unparsed issue_created values:
Series([], )


In [17]:
# Rebuild creation-time features using mixed timestamp parsing

model_df = priority_issue_df.copy()

model_df["issue_created"] = pd.to_datetime(
    model_df["issue_created"],
    utc=True,
    errors="coerce",
    format="mixed",
)

model_df["created_year"] = model_df["issue_created"].dt.year
model_df["created_month"] = model_df["issue_created"].dt.month
model_df["created_dayofweek"] = model_df["issue_created"].dt.dayofweek
model_df["created_hour"] = model_df["issue_created"].dt.hour

feature_cols = [
    "issue_proj",
    "issue_type",
    "created_year",
    "created_month",
    "created_dayofweek",
    "created_hour",
]

X = model_df[feature_cols].copy()
y = model_df["issue_priority"].copy()

print("X shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum())

X shape: (32661, 6)

Missing values:
issue_proj           0
issue_type           0
created_year         0
created_month        0
created_dayofweek    0
created_hour         0
dtype: int64


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ---------------------------------------------------------
# Train / test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain target distribution")
print(y_train.value_counts())

print("\nTest target distribution")
print(y_test.value_counts())

# ---------------------------------------------------------
# Majority-class baseline
# ---------------------------------------------------------
majority_class = y_train.mode()[0]
majority_pred = [majority_class] * len(y_test)

baseline_accuracy = accuracy_score(y_test, majority_pred)
baseline_macro_f1 = f1_score(
    y_test,
    majority_pred,
    average="macro",
    zero_division=0,
)

print("\nMajority baseline")
print("=" * 50)
print("Majority class :", majority_class)
print("Accuracy       :", round(baseline_accuracy, 4))
print("Macro-F1       :", round(baseline_macro_f1, 4))

# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------
categorical_features = [
    "issue_proj",
    "issue_type",
]

numeric_features = [
    "created_year",
    "created_month",
    "created_dayofweek",
    "created_hour",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent"),
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            min_frequency=2,
                        ),
                    ),
                ]
            ),
            categorical_features,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="median"),
                    ),
                ]
            ),
            numeric_features,
        ),
    ]
)

# ---------------------------------------------------------
# Class-balanced Logistic Regression
# ---------------------------------------------------------
priority_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
priority_model.fit(X_train, y_train)

# ---------------------------------------------------------
# Evaluate
# ---------------------------------------------------------
y_pred = priority_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)
weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0,
)

print("\nPriority Detection — Logistic Regression")
print("=" * 50)
print("Accuracy    :", round(accuracy, 4))
print("Macro-F1    :", round(macro_f1, 4))
print("Weighted-F1 :", round(weighted_f1, 4))

print("\nClassification report")
print("=" * 50)
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0,
    )
)

Train: (26128, 6)
Test : (6533, 6)

Train target distribution
issue_priority
Medium     19828
High        3595
Highest     1667
Blocker      523
Low          448
Lowest        67
Name: count, dtype: int64

Test target distribution
issue_priority
Medium     4958
High        898
Highest     417
Blocker     131
Low         112
Lowest       17
Name: count, dtype: int64

Majority baseline
Majority class : Medium
Accuracy       : 0.7589
Macro-F1       : 0.1438

Priority Detection — Logistic Regression
Accuracy    : 0.3998
Macro-F1    : 0.2349
Weighted-F1 : 0.4904

Classification report
              precision    recall  f1-score   support

     Blocker       0.08      0.44      0.13       131
        High       0.27      0.36      0.31       898
     Highest       0.23      0.35      0.28       417
         Low       0.06      0.54      0.11       112
      Lowest       0.01      0.47      0.02        17
      Medium       0.90      0.41      0.56      4958

    accuracy                     

c:\Users\danie\anaconda3\envs\supportmind\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

# Add one safe ticket-level feature
X_tree = X.copy()
X_tree["issue_contr_count"] = priority_issue_df["issue_contr_count"].values

categorical_features = [
    "issue_proj",
    "issue_type",
]

numeric_features = [
    "created_year",
    "created_month",
    "created_dayofweek",
    "created_hour",
    "issue_contr_count",
]

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2,
            ),
            categorical_features,
        ),
        (
            "num",
            "passthrough",
            numeric_features,
        ),
    ]
)

tree_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            ExtraTreesClassifier(
                n_estimators=400,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=2,
            ),
        ),
    ]
)

tree_model.fit(X_train.assign(
    issue_contr_count=priority_issue_df.loc[X_train.index, "issue_contr_count"]
), y_train)

X_test_tree = X_test.copy()
X_test_tree["issue_contr_count"] = priority_issue_df.loc[
    X_test.index, "issue_contr_count"
]

y_pred_tree = tree_model.predict(X_test_tree)

print("ExtraTrees Priority Detection")
print("=" * 50)
print("Accuracy    :", round(accuracy_score(y_test, y_pred_tree), 4))
print(
    "Macro-F1    :",
    round(f1_score(y_test, y_pred_tree, average="macro", zero_division=0), 4),
)
print(
    "Weighted-F1 :",
    round(
        f1_score(y_test, y_pred_tree, average="weighted", zero_division=0),
        4,
    ),
)

print("\nClassification report")
print(
    classification_report(
        y_test,
        y_pred_tree,
        zero_division=0,
    )
)

ExtraTrees Priority Detection
Accuracy    : 0.7326
Macro-F1    : 0.3966
Weighted-F1 : 0.7635

Classification report
              precision    recall  f1-score   support

     Blocker       0.19      0.44      0.26       131
        High       0.46      0.48      0.47       898
     Highest       0.34      0.50      0.40       417
         Low       0.18      0.52      0.27       112
      Lowest       0.06      0.35      0.10        17
      Medium       0.95      0.81      0.87      4958

    accuracy                           0.73      6533
   macro avg       0.36      0.52      0.40      6533
weighted avg       0.81      0.73      0.76      6533



In [22]:
from pathlib import Path
import json
import joblib

MODEL_DIR = Path(
    r"C:\Users\danie\supportmind-ai\backend\app\nlp\priority_detection"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "priority_detection_model.joblib"
METADATA_PATH = MODEL_DIR / "model_metadata.json"

# ---------------------------------------------------------
# Save the complete pipeline
# ---------------------------------------------------------
joblib.dump(tree_model, MODEL_PATH)

metadata = {
    "model": "ExtraTreesClassifier",
    "task": "Priority Detection",
    "dataset": "issues.csv",
    "target": "issue_priority",
    "excluded_target_value": "unknown",
    "issue_key": ["issue_proj", "issue_num"],
    "feature_columns": [
        "issue_proj",
        "issue_type",
        "created_year",
        "created_month",
        "created_dayofweek",
        "created_hour",
        "issue_contr_count",
    ],
    "n_estimators": 400,
    "class_weight": "balanced",
    "min_samples_leaf": 2,
    "random_state": 42,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "accuracy": float(
        accuracy_score(y_test, y_pred_tree)
    ),
    "macro_f1": float(
        f1_score(
            y_test,
            y_pred_tree,
            average="macro",
            zero_division=0,
        )
    ),
    "weighted_f1": float(
        f1_score(
            y_test,
            y_pred_tree,
            average="weighted",
            zero_division=0,
        )
    ),
    "classes": tree_model.named_steps["classifier"].classes_.tolist(),
}

METADATA_PATH.write_text(
    json.dumps(metadata, indent=4),
    encoding="utf-8",
)

print("Model saved:")
print(MODEL_PATH)

print("\nMetadata saved:")
print(METADATA_PATH)

Model saved:
C:\Users\danie\supportmind-ai\backend\app\nlp\priority_detection\priority_detection_model.joblib

Metadata saved:
C:\Users\danie\supportmind-ai\backend\app\nlp\priority_detection\model_metadata.json


In [20]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

priority_order = {
    "Lowest": 0,
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Highest": 4,
    "Blocker": 5,
}

inverse_priority = {v: k for k, v in priority_order.items()}

y_train_ord = y_train.map(priority_order)
y_test_ord = y_test.map(priority_order)

ordinal_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "regressor",
            ExtraTreesRegressor(
                n_estimators=400,
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=2,
            ),
        ),
    ]
)

X_train_ord = X_train.assign(
    issue_contr_count=priority_issue_df.loc[
        X_train.index, "issue_contr_count"
    ]
)

X_test_ord = X_test.assign(
    issue_contr_count=priority_issue_df.loc[
        X_test.index, "issue_contr_count"
    ]
)

ordinal_model.fit(X_train_ord, y_train_ord)

pred_numeric = ordinal_model.predict(X_test_ord)

# Convert continuous prediction to nearest priority level
pred_numeric = pred_numeric.round().clip(0, 5).astype(int)

y_pred_ordinal = pd.Series(
    pred_numeric,
    index=y_test.index,
).map(inverse_priority)

print("Ordinal ExtraTrees Priority Detection")
print("=" * 50)

print(
    "Accuracy    :",
    round(accuracy_score(y_test, y_pred_ordinal), 4),
)

print(
    "Macro-F1    :",
    round(
        f1_score(
            y_test,
            y_pred_ordinal,
            average="macro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Weighted-F1 :",
    round(
        f1_score(
            y_test,
            y_pred_ordinal,
            average="weighted",
            zero_division=0,
        ),
        4,
    ),
)

print("\nClassification report")
print(
    classification_report(
        y_test,
        y_pred_ordinal,
        zero_division=0,
    )
)

Ordinal ExtraTrees Priority Detection
Accuracy    : 0.784
Macro-F1    : 0.3312
Weighted-F1 : 0.7764

Classification report
              precision    recall  f1-score   support

     Blocker       0.82      0.07      0.13       131
        High       0.42      0.56      0.48       898
     Highest       0.38      0.27      0.32       417
         Low       0.32      0.11      0.16       112
      Lowest       0.00      0.00      0.00        17
      Medium       0.90      0.90      0.90      4958

    accuracy                           0.78      6533
   macro avg       0.47      0.32      0.33      6533
weighted avg       0.79      0.78      0.78      6533



In [23]:
from pathlib import Path
import json
import joblib

MODEL_DIR = Path(
    r"C:\Users\danie\supportmind-ai\backend\app\nlp\priority_detection"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "priority_detection_model.joblib"
METADATA_PATH = MODEL_DIR / "model_metadata.json"

# ---------------------------------------------------------
# Save the complete pipeline
# ---------------------------------------------------------
joblib.dump(tree_model, MODEL_PATH)

metadata = {
    "model": "ExtraTreesClassifier",
    "task": "Priority Detection",
    "dataset": "issues.csv",
    "target": "issue_priority",
    "excluded_target_value": "unknown",
    "issue_key": ["issue_proj", "issue_num"],
    "feature_columns": [
        "issue_proj",
        "issue_type",
        "created_year",
        "created_month",
        "created_dayofweek",
        "created_hour",
        "issue_contr_count",
    ],
    "n_estimators": 400,
    "class_weight": "balanced",
    "min_samples_leaf": 2,
    "random_state": 42,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "accuracy": float(
        accuracy_score(y_test, y_pred_tree)
    ),
    "macro_f1": float(
        f1_score(
            y_test,
            y_pred_tree,
            average="macro",
            zero_division=0,
        )
    ),
    "weighted_f1": float(
        f1_score(
            y_test,
            y_pred_tree,
            average="weighted",
            zero_division=0,
        )
    ),
    "classes": tree_model.named_steps["classifier"].classes_.tolist(),
}

METADATA_PATH.write_text(
    json.dumps(metadata, indent=4),
    encoding="utf-8",
)

print("Model saved:")
print(MODEL_PATH)

print("\nMetadata saved:")
print(METADATA_PATH)

Model saved:
C:\Users\danie\supportmind-ai\backend\app\nlp\priority_detection\priority_detection_model.joblib

Metadata saved:
C:\Users\danie\supportmind-ai\backend\app\nlp\priority_detection\model_metadata.json


In [24]:
import joblib
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

MODEL_PATH = r"C:\Users\danie\supportmind-ai\backend\app\nlp\priority_detection\priority_detection_model.joblib"

reloaded_model = joblib.load(MODEL_PATH)

reloaded_pred = reloaded_model.predict(X_test_tree)

same_predictions = np.array_equal(
    y_pred_tree,
    reloaded_pred,
)

reload_accuracy = accuracy_score(y_test, reloaded_pred)

reload_macro_f1 = f1_score(
    y_test,
    reloaded_pred,
    average="macro",
    zero_division=0,
)

reload_weighted_f1 = f1_score(
    y_test,
    reloaded_pred,
    average="weighted",
    zero_division=0,
)

print("Reload verification")
print("=" * 50)
print("Predictions identical :", same_predictions)
print("Accuracy              :", round(reload_accuracy, 4))
print("Macro-F1              :", round(reload_macro_f1, 4))
print("Weighted-F1           :", round(reload_weighted_f1, 4))

Reload verification
Predictions identical : True
Accuracy              : 0.7326
Macro-F1              : 0.3966
Weighted-F1           : 0.7635


In [25]:
import pandas as pd
import joblib

# ---------------------------------------------------------
# Load the saved Priority Detection model
# ---------------------------------------------------------
MODEL_PATH = (
    r"C:\Users\danie\supportmind-ai\backend\app\nlp"
    r"\priority_detection\priority_detection_model.joblib"
)

reloaded_model = joblib.load(MODEL_PATH)

# ---------------------------------------------------------
# Create completely new ticket examples
# ---------------------------------------------------------
new_tickets = pd.DataFrame([
    {
        "issue_proj": "C06ggn039",
        "issue_type": "Ticket",
        "created_year": 2026,
        "created_month": 9,
        "created_dayofweek": 3,
        "created_hour": 10,
        "issue_contr_count": 1,
    },
    {
        "issue_proj": "C06ggn039",
        "issue_type": "Deployment",
        "created_year": 2026,
        "created_month": 9,
        "created_dayofweek": 3,
        "created_hour": 14,
        "issue_contr_count": 2,
    },
    {
        "issue_proj": "C07gg6cz0",
        "issue_type": "Bug",
        "created_year": 2026,
        "created_month": 9,
        "created_dayofweek": 4,
        "created_hour": 9,
        "issue_contr_count": 1,
    },
    {
        "issue_proj": "C12hhx18aj28av.ew8x3d",
        "issue_type": "Ticket",
        "created_year": 2026,
        "created_month": 9,
        "created_dayofweek": 4,
        "created_hour": 16,
        "issue_contr_count": 3,
    },
])

# ---------------------------------------------------------
# Predict
# ---------------------------------------------------------
predictions = reloaded_model.predict(new_tickets)

probabilities = reloaded_model.predict_proba(new_tickets)

# ---------------------------------------------------------
# Build inference output
# ---------------------------------------------------------
inference_output = new_tickets.copy()

inference_output["predicted_priority"] = predictions

inference_output["confidence"] = (
    probabilities.max(axis=1).round(4)
)

# Optional: show the probability of every class
class_names = reloaded_model.named_steps["classifier"].classes_

for i, class_name in enumerate(class_names):
    inference_output[f"prob_{class_name}"] = (
        probabilities[:, i].round(4)
    )

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------
print("Final Priority Detection Inference Test")
print("=" * 70)

print(
    inference_output.to_string(index=False)
)

Final Priority Detection Inference Test
           issue_proj issue_type  created_year  created_month  created_dayofweek  created_hour  issue_contr_count predicted_priority  confidence  prob_Blocker  prob_High  prob_Highest  prob_Low  prob_Lowest  prob_Medium
            C06ggn039     Ticket          2026              9                  3            10                  1               High      0.2706        0.0554     0.2706        0.2602    0.1392       0.0081       0.2665
            C06ggn039 Deployment          2026              9                  3            14                  2            Blocker      0.2512        0.2512     0.2339        0.2494    0.0887       0.0130       0.1638
            C07gg6cz0        Bug          2026              9                  4             9                  1             Medium      0.3540        0.0718     0.2299        0.1734    0.1394       0.0315       0.3540
C12hhx18aj28av.ew8x3d     Ticket          2026              9                  4